In [2]:
from google.colab import drive
drive.mount('/content/drive')
!pip install keras nltk scikit-learn python-docx
from docx import Document
import nltk
from nltk import tokenize
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.lm.preprocessing import padded_everygram_pipeline

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 6.4 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


# Passo 1. Carregar o Corpus

In [3]:
docx_file_path = '/content/drive/MyDrive/UNIVESP/PI4/PESQUISA MISOGINIA NO FUNK/Letras/TOP músicas 2025 - letras.docx'

try:
    document = Document(docx_file_path)
    full_text = []
    for paragraph in document.paragraphs:
        full_text.append(paragraph.text)

    # Join the paragraphs into a single string or keep as a list of strings
    # Here, we'll join them, similar to how the 'poema' was handled initially
    poema = '\n'.join(full_text)
    print("Text extracted successfully:")
    print(poema[:1000]) # Print first 500 characters to verify

except FileNotFoundError:
    print(f"Error: The file '{docx_file_path}' was not found. Please check the path.")
except Exception as e:
    print(f"An error occurred: {e}")

Text extracted successfully:
Lista de Músicas 5
P Do Pecado (Ao Vivo) – Grupo Menos É Mais, Simone Mendes
Uma ligação fria com um convite quente
Você é uma comprometida que me liga
E eu sou o solteiro que atende
Depois que o suor seca no chão
É fácil me chamar de tentação
Como se eu fosse o culpado
Do que cê faz com o seu namorado
Como se eu que tivesse alguém
Mas diferente de você
Meu beijo não tá enganando ninguém
Jogando erro contra erro
O seu é bem maior que o meu
Da vida, cê não tira ele
Da cama, cê não tira eu
Não joga a culpa toda pro meu lado
É você que tem alguém do lado
Eu sou apenas o P do pecado
Eu sou apenas o P do pecado
Jogando erro contra erro
O seu é bem maior que o meu
Da vida, cê não tira ele
Da cama, cê não tira eu
Não joga a culpa toda pro meu lado
É você que tem alguém do lado
Eu sou apenas o P do pecado
Eu sou apenas o P do pecado
Chegamo' no pagode
Chegamo' no pagode em
Daquele jeito (esqueça)
Pagode de primeira, daquele jeito
Se precisar nós resolve, tá tudo ce

# Passo 2. Tokenizando o corpus


In [ ]:
poema = poema.lower().split("\n")
print(poema[:500])

In [ ]:
texto_tok = []
for verso in poema:
  tokens = tokenize.word_tokenize(verso, language="portuguese")
  texto_tok.append(tokens)

print(texto_tok[:500])

# Simplificando o processo (passos 3 a 6 com um só comando)
* Passo 3. Inserindo os marcadores de início e fim de sentença
* Passo 4. Calculando os n-gramas
* Passo 5. Colocando todos os tokens do corpus em uma única lista
* Passo 6. Definindo o vocabulário





In [ ]:
ngrams = 3
ngramas_pad, vocab = padded_everygram_pipeline(ngrams, texto_tok)

#Remoção de stopwords

In [ ]:
nltk.download('stopwords')
stopwords = nltk.corpus.stopwords.words('portuguese')

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words=stopwords)

vetores = vectorizer.fit_transform(poema)
vocab = vectorizer.get_feature_names_out()

print("Vocabulário\n", vocab , "\n")
print("Matriz\n", vetores.toarray())

# Gerando *word embeddings*
Agora estamos prontos para gerar nossas WEs a partir dessas sentenças. Para isso, basta executar o código abaixo, que gera WEs usando word2vec (um dos algoritmos mais tradicionais para geração de WEs) com dimensão 300 (a mesma das WEs do NILC).

In [ ]:
#!pip install torch pot gensim transformers
from gensim.models import Word2Vec
meu_modelo = Word2Vec(texto_tok,
                 min_count=3,     # Ignora palavras que ocorrem menos de 3 vezes
                 vector_size=300, # Dimensão das embeddings
                 workers=2,       # Número de processadores (paralelização)
                 window=5,        # Tamanho da janela de contexto, em palavras, usada durante o treinamento
                 epochs=30)       # Número de épocas de treinamento

len(meu_modelo.wv.index_to_key)

meu_modelo.wv.most_similar("puta")

texto_tok

# Salvando as *word embeddings* geradas

Por fim, podemos salvar o modelo treinado para ser carregado para uso futuro (de modo similar ao que fizemos na parte 1 deste código). É importante salvar o modelo, pois o CoLab vai remover tudo assim que esta seção terminar.

In [ ]:
#meu_modelo.wv.save_word2vec_format('meu_modelo.bin', binary=True)